# Milestone 5

## Q1): Dataset Splitting Strategy Task: You are building a lazy-loading "recipe" generator for the 10 genres.

Instead of loading audio into RAM, you will create a list of dictionaries. Assume you create exactly 100 recipe dictionaries per genre. Pass this combined list of recipes into sklearn.model_selection.train_test_split using test_size=0.2, shuffle=True, and random_state=42.

Question: Exactly how many recipe items will be allocated to your validation set (val_recipes) list?

In [1]:
from sklearn.model_selection import train_test_split

# Create dummy recipe list (100 per genre × 10 genres)
genres = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]

recipes = []

for genre in genres:
    for i in range(100):
        recipes.append({
            "genre": genre,
            "id": f"{genre}_{i}"
        })

# Split
train_recipes, val_recipes = train_test_split(
    recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("Total:", len(recipes))
print("Validation:", len(val_recipes))

Total: 1000
Validation: 200


## Q2) On-the-Fly Mixing Dimensions Task: In your training loop, you will load audio stems on the fly. 

Assume you use librosa.load(sr=16000, duration=10) to load 4 instrument stems and 1 noise file. You pad or truncate all 5 arrays to be exactly 160,000 samples long. 

You sum the 4 stems together to create a mix array, and then add the noise array scaled by an intensity weight of 0.2. 


Question: Before this final mix array is passed to the feature extractor, what must its exact 1D NumPy shape be? 

In [2]:
import numpy as np

# Simulate 5 audio arrays
drums = np.ones(160000)
vocals = np.ones(160000)
bass = np.ones(160000)
others = np.ones(160000)
noise = np.ones(160000)

# Mix stems
mix = drums + vocals + bass + others

# Add noise
mix = mix + 0.2 * noise

print("Shape:", mix.shape)

Shape: (160000,)


## Question 3: Hugging Face Feature Extractor Shape Task: 


Initialize Hugging Face's AutoFeatureExtractor using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

To test your pipeline, create a dummy audio mix of exactly 160,000 ones using 
mix = np.ones(160000). 
Pass this dummy array into the extractor with sampling_rate=16000 and return_tensors="pt". 
Finally, extract the input_values from the resulting dictionary and apply .squeeze(0) to remove the default batch dimension. 


Question: What is the exact shape of the resulting PyTorch tensor? (This is the spectrogram matrix the AST model will process). In this format: [a,b] or (a,b)

In [3]:
import numpy as np
from transformers import AutoFeatureExtractor

# Load extractor
extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

# Dummy audio
mix = np.ones(160000)

# Extract features
inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

spectrogram = inputs["input_values"].squeeze(0)

print("Shape:", spectrogram.shape)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Shape: torch.Size([1024, 128])


## Q4: Model Architecture Initialization Task: 
Initialize the ASTForAudioClassification.from_pretrained model using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

Because we are classifying 10 genres instead of the original AudioSet classes, you must explicitly pass num_labels=10 and ignore_mismatched_sizes=True to replace the classification head. 

Question: Write a quick loop using 

sum(p.numel() for p in model.parameters() if p.requires_grad) 

to calculate the number of trainable parameters in this newly configured model. What is the exact integer value?



In [4]:
from transformers import ASTForAudioClassification

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters:", num_params)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable parameters: 86196490


## Q5): Inference Normalization Math Task: In your test inference loop, right before feature extraction, the audio is normalized to prevent clipping using this exact formula: 

y = y / (np.max(np.abs(y)) + 1e-9). 

To test this, create a NumPy array: 
y_test = np.array([-0.85, 0.40, 0.20, -0.10]). 

Apply the normalization formula to it. 

Question: What is the resulting value at index 0 of the normalized array, rounded to 3 decimal places?

In [6]:
import numpy as np

y_test = np.array([-0.85, 0.40, 0.20, -0.10])

y_norm = y_test / (np.max(np.abs(y_test)) + 1e-9)

print("Normalized array:", y_norm)
print("Value at index 0:", round(y_norm[0], 3))

Normalized array: [-1.          0.47058823  0.23529412 -0.11764706]
Value at index 0: -1.0
